<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone: Which Content Pages Should You Refresh First?

This notebook mirrors the deployed research paper section by section. It's a compact,
self-contained rerun of the pipeline built across w02–w07: the same population, the same
honest client-grouped validation, the same baseline, model, and playbook condensed to what
the paper needs to show and cite.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

Working dir: /content/Applied-Search-Intelligence-System/Applied-Search-Intelligence-System


## 1. Question

**Do the hand‑picked refresh rules really predict decline, or can a validated model do better?**

The decision this supports: a content team facing thousands of pages and limited editorial time needs to know which pages to look at *first* in a refresh cycle. The common approach is an informal rule ("it's old, it still gets traffic, fix it") applied by eye. This project checks
whether that kind of rule holds up under real measurement, builds a transparent version of it as a baseline, and tests whether a validated model earns the right to replace it.

## 2. Data

**Source:** the FlyRank ML Internship starter dataset, `data/raw/content_refresh_anonymized.csv`, 30,000 pseudonymized content pages across 32 clients, one row per page, with trailing-90-day
search (GSC-style) and engagement (GA4-style) metrics. A larger 78.8M row daily-grain warehouse
also exists (queried in my w03 data-contract notebook to verify grain and schema); this project uses the starter CSV because its columns match the feature set this lane's baseline and model were built around.

**Population filter:**
 `impressions_90d > 0` (the page has real search visibility to judge)

 and

`content_age_days >= 90` (the page is old enough to have a genuine 90-day trend).

On this dataset every one of the 30,000 rows already satisfies both conditions.


**Excluded, on purpose, and why:**

`trend_direction` and `trend_pct` (the columns the label is
built from), and the raw `*_last_30d` / `*_prev_30d` windows (the columns those are built from);
all excluded from every feature set in this project to avoid leakage. Confirmed empirically in section 3 below.

**Public-safety:** `client_id` and `content_id` are already pseudonymized hashes in the source
data; they're used only for grouping (never as model features), and no client names, URLs, or
raw search queries appear anywhere in this project.

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW_PATH)
initial_rows = len(df)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {len(df):,} of {initial_rows:,} raw rows")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Base rate (share declining): {df['is_declining_label'].mean():.3f}")

Rows: 30,000 of 30,000 raw rows
Unique clients: 32
Base rate (share declining): 0.542


## 3. Methodology

**Label:** `is_declining_label = 1` where `trend_direction == "down"` ,  a proxy, not a directly
observed outcome; it's a rule on a 30-day-vs-prior-30-day impression comparison, and w05's error
analysis found it genuinely noisy at low impression counts.

**Baseline (from w04):** a transparent two-gate rule, `impressions_90d >= 500` (a reliability
floor, motivated by a signal check showing raw traffic volume doesn't itself predict decline) AND `days_since_last_update >= 90` (the threshold the large-sample buckets actually confirmed: 61% decline at 91-180 days vs. 51% for pages updated in the last 30). A below-tier-median CTR adds a secondary bonus. Building this rule required fixing a real data trap first: rows with `avg_position == 0` ("no position data") were silently sorting into the `top_3` tier and would have corrupted any CTR-vs-position comparison.

**Model:** Logistic Regression over 22 numeric and 10 categorical features (current-state
90-day totals and tiers only, never a `last_30d`/`prev_30d` window). A Random Forest was
trained alongside it as a stronger-but-opaque comparison.

**Validation design:** `GroupShuffleSplit` (80/20) grouped by `client_id`,  zero client overlap
between train and test. This matters because a random row split let 31 of 32 clients leak into
both sides, and inflated every metric (ROC-AUC 0.704 vs. the grouped split's honest 0.577;
precision@20 0.900 vs. 0.650) as shown side-by-side below.

**Leakage checks:** deliberately adding `trend_pct` (the literal label source) to the honest feature set pushed ROC-AUC from 0.577 to 0.989; adding the raw 30-day windows pushed it to 0.836, confirming both that the checking harness works and that the real feature set is clean.

In [13]:
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc", "word_count", "char_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier", "position_tier",
    "provider_used", "model_used",
]
X = df[numeric_features + categorical_features]
y = df["is_declining_label"]
groups = df["client_id"]

def make_pipe(model):
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    return Pipeline([("prep", preprocess), ("clf", model)])

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores)
    return np.asarray(labels)[order][:k].mean()

# Random vs grouped split -- the before/after that motivates the whole validation design.
rs = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_r, te_r = next(rs.split(X, y))
pipe_r = make_pipe(LogisticRegression(max_iter=2000, random_state=42)).fit(X.iloc[tr_r], y.iloc[tr_r])
s_r = pipe_r.predict_proba(X.iloc[te_r])[:, 1]
y_r = y.iloc[te_r].to_numpy()

gs = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gs.split(X, y, groups=groups))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]
df_train, df_test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)

print("Random split - ROC-AUC:", round(roc_auc_score(y_r, s_r), 3), " precision@20:", round(precision_at_k(s_r, y_r, 20), 3))
print(f"  ({len(set(df.iloc[tr_r].client_id) & set(df.iloc[te_r].client_id))} of {groups.nunique()} clients leak across train/test)")

Random split - ROC-AUC: 0.704  precision@20: 0.9
  (31 of 32 clients leak across train/test)


## 4. Results (vs baseline)

Baseline rule, Logistic Regression, and Random Forest, all scored on the **same client-grouped
held-out slice**, the honest comparison the whole pipeline was built to produce.

In [14]:
# Baseline rule, re-scored on the test slice only (tier medians fit on TRAIN only).
usable_tiers = ["page_1", "page_3_5", "striking"]
pos_known_mask = df_train["avg_position"] > 0
tier_median_map = (
    df_train.loc[pos_known_mask & df_train["position_tier"].isin(usable_tiers)]
    .groupby("position_tier")["ctr"].median()
)

def score_with_rule(frame):
    frame = frame.copy()
    frame["tier_median_ctr"] = frame["position_tier"].map(tier_median_map)
    visible = frame["impressions_90d"] >= 500
    stale = frame["days_since_last_update"] >= 90
    ctr_gap = (
        (frame["avg_position"] > 0) & frame["position_tier"].isin(usable_tiers)
        & (frame["ctr"] < frame["tier_median_ctr"])
    )
    gate = visible & stale
    bonus = np.where(ctr_gap, 0.5, 0.0)
    return np.where(gate, (1 + bonus) * np.log1p(frame["impressions_90d"]), 0.0)

baseline_score = score_with_rule(df_test)

logreg = make_pipe(LogisticRegression(max_iter=2000, random_state=42)).fit(X_train, y_train)
rf = make_pipe(RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)).fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]
rf_score = rf.predict_proba(X_test)[:, 1]
y_test_arr = y_test.to_numpy()

rows = []
for name, scores in [("Week-4 rule baseline", baseline_score), ("Logistic Regression", logreg_score), ("Random Forest", rf_score)]:
    row = {"model": name}
    for k in [20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(scores, y_test_arr, k), 3)
    row["ROC-AUC"] = round(roc_auc_score(y_test_arr, scores), 3)
    row["Avg Precision"] = round(average_precision_score(y_test_arr, scores), 3)
    rows.append(row)
results_table = pd.DataFrame(rows).set_index("model")
results_table.loc["(test base rate)"] = [round(y_test_arr.mean(), 3)] * 3 + [np.nan, np.nan]
results_table

,precision@20,precision@50,precision@100,ROC-AUC,Avg Precision
model,,,,,
Week-4 rule baseline,0.500,0.520,0.470,0.489,0.508
Logistic Regression,0.650,0.640,0.670,0.577,0.567
Random Forest,0.600,0.580,0.530,0.602,0.582
(test base rate),0.511,0.511,0.511,NaN,NaN


**Reading it straight:** the rule that looked strong on its own tuning population in week 4
(precision@20 of 0.650, measured on all 30,000 rows) drops to base rate on 7 completely unseen
clients here (0.500). Logistic Regression wins at every precision@K cut that matters for a queue
an editor works top-down; Random Forest wins on whole-ranking measures (ROC-AUC, Avg Precision)
but loses to Logistic Regression specifically where it counts. The simpler, more readable model
is the one this project recommends.

In [15]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Path("work/figures").mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4.5))
models = ["Week-4 rule\nbaseline", "Logistic\nRegression", "Random\nForest"]
p20 = [results_table.loc["Week-4 rule baseline", "precision@20"],
       results_table.loc["Logistic Regression", "precision@20"],
       results_table.loc["Random Forest", "precision@20"]]
colors = ["#8a8378", "#1f6f5c", "#5b8a99"]
bars = ax.bar(models, p20, color=colors, width=0.55)
ax.axhline(y_test_arr.mean(), color="#b5442e", linestyle="--", linewidth=1.5, label=f"test base rate ({y_test_arr.mean():.3f})")
for bar, v in zip(bars, p20):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("precision@20 (held-out, client-grouped)")
ax.set_ylim(0, 0.8)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/model_vs_baseline.png", dpi=150)
plt.close()
print("Saved work/figures/model_vs_baseline.png")

Saved work/figures/model_vs_baseline.png


In [16]:
# The before/after that justifies the whole validation design -- random vs grouped split.
fig, ax = plt.subplots(figsize=(7, 4.5))
splits = ["Random split\n(naive)", "Client-grouped split\n(honest)"]
aucs = [round(roc_auc_score(y_r, s_r), 3), results_table.loc["Logistic Regression", "ROC-AUC"]]
p20s = [round(precision_at_k(s_r, y_r, 20), 3), results_table.loc["Logistic Regression", "precision@20"]]
x = np.arange(2)
width = 0.32
ax.bar(x - width/2, aucs, width, label="ROC-AUC", color="#1f6f5c")
ax.bar(x + width/2, p20s, width, label="precision@20", color="#c98a2e")
ax.set_xticks(x); ax.set_xticklabels(splits)
for i, (a, p) in enumerate(zip(aucs, p20s)):
    ax.text(i - width/2, a + 0.02, f"{a:.3f}", ha="center", fontsize=9)
    ax.text(i + width/2, p + 0.02, f"{p:.3f}", ha="center", fontsize=9)
ax.set_ylim(0, 1.0)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Same model, same data -- split design alone moves the score")
plt.tight_layout()
plt.savefig("work/figures/split_before_after.png", dpi=150)
plt.close()
print("Saved work/figures/split_before_after.png")

Saved work/figures/split_before_after.png


## 5. Limitations

- **Proxy label, not ground truth.** `is_declining_label` is a threshold on a noisy 30-day swing;
  w05's error analysis found a false negative built on a single impression.
- **One held-out split.** These numbers come from one client-grouped 80/20 partition. A single
  split is real evidence, not a guarantee,  repeated group splits would strengthen the claim.
- **Cross-sectional, one snapshot.** Nothing here supports "refreshing this page will increase
  its traffic." The honest form is decision-support: this page looks worth reviewing first.
- **Not yet validated on a brand-new client.** The grouped-split gap above is itself evidence
  that performance on a client outside this population should be treated cautiously until
  locally checked.
- **Does not model a search engine's ranking algorithm.** It models this portfolio's own
  observed performance patterns, nothing more.
- **No claims about specific AI writing tools.** `model_used` and `content_type` showed up as
  model features but sample sizes per category are small and uneven, not evidence about any
  one tool's content quality (see the action playbook's no-go list).

## 6. Ranked recommendations

The full action playbook (w07) retrains the validated Logistic Regression on the complete
population, attaches a reason code restricted to actionable features, sorts by a **value-weighted
score** `(probability * click-equivalent value)` rather than raw probability, and groups pages into
four descriptive archetypes. The headline finding: the top 5 pages by raw decline probability all
had **zero clicks in 90 days**, pages worth nothing economically while the value-weighted
queue surfaces a completely different, non-overlapping set of pages that actually carry click and CPC value at risk.

In [17]:
archetype_summary = pd.DataFrame({
    "Steady & Visible": [20113, "67%", "Routine monitoring"],
    "Visible But Stale": [9269, "31%", "Primary refresh target"],
    "High-Engagement Niche": [486, "1.6%", "Protect & study"],
    "Near-Zero-Data Edge Cases": [132, "0.4%", "Exclude from automated scoring"],
}, index=["pages", "share", "recommended angle"]).T
archetype_summary

,pages,share,recommended angle
Steady & Visible,20113,67%,Routine monitoring
Visible But Stale,9269,31%,Primary refresh target
High-Engagement Niche,486,1.6%,Protect & study
Near-Zero-Data Edge Cases,132,0.4%,Exclude from automated scoring


## 7. Artifacts the paper embeds

Both figures generated in section 4 (`model_vs_baseline.png`, `split_before_after.png`) plus the
action-tier/archetype figure already committed from w07 (`action_playbook_overview.png`) are the
three charts the deployed paper uses. The action playbook's queue and metrics JSON
(`work/outputs/action_playbook_queue.csv`, `work/outputs/action_playbook_metrics.json`) are the
receipts the paper's recommendation numbers trace back to.

In [18]:
import json
Path("work/outputs").mkdir(parents=True, exist_ok=True)
capstone_metrics = {
    "population": {"rows": int(len(df)), "clients": int(df["client_id"].nunique()), "base_rate": round(float(df["is_declining_label"].mean()), 3)},
    "results_vs_baseline_grouped_split": results_table.reset_index().to_dict("records"),
    "split_before_after": {"random_split_auc": round(roc_auc_score(y_r, s_r), 3), "random_split_precision_at_20": round(precision_at_k(s_r, y_r, 20), 3),
                             "grouped_split_auc": results_table.loc["Logistic Regression", "ROC-AUC"], "grouped_split_precision_at_20": results_table.loc["Logistic Regression", "precision@20"]},
    "leakage_check": {"clean_auc": 0.577, "with_trend_pct_auc": 0.989, "with_30d_windows_auc": 0.836},
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2, default=str)
print("Wrote work/outputs/capstone_metrics.json")
print(json.dumps(capstone_metrics, indent=2, default=str)[:600])

Wrote work/outputs/capstone_metrics.json
{
  "population": {
    "rows": 30000,
    "clients": 32,
    "base_rate": 0.542
  },
  "results_vs_baseline_grouped_split": [
    {
      "model": "Week-4 rule baseline",
      "precision@20": 0.5,
      "precision@50": 0.52,
      "precision@100": 0.47,
      "ROC-AUC": 0.489,
      "Avg Precision": 0.508
    },
    {
      "model": "Logistic Regression",
      "precision@20": 0.65,
      "precision@50": 0.64,
      "precision@100": 0.67,
      "ROC-AUC": 0.577,
      "Avg Precision": 0.567
    },
    {
      "model": "Random Forest",
      "precision@20": 0.6,
      "precision@50": 0.58,
  


## Self-check

- [x] Every section mirrors a section of the deployed paper
- [x] Model vs baseline on the same client-grouped split, with the before/after split comparison
- [x] Leakage checks re-stated with real numbers
- [x] Limitations stated in safe, claim-ladder language
- [x] Ranked recommendations summarized from w07
- [x] Figures and metrics exported for the paper to embed
- [x] Committed to my repo, paper deployed, `submission/paper_url.txt` updated — Done.